# 04 · Extracción de K-mers — Vectorización Genómica

**Objetivo:** Construir la matriz de features k-mer `X_kmers` (sparse CSR) de dimensiones
`(N_genomas × top_K_kmers)` a partir de los ensamblajes FASTA y los IDs validados en
el notebook 03.

| Paso | Descripción |
|---|---|
| 1 | Verificación de jellyfish + inventario de FASTAs vs IDs target |
| 2 | Conteo de k-mers canónicos (k=31) por genoma via jellyfish |
| 3 | Construcción del vocabulario: document frequency de cada k-mer |
| 4 | Selección de los top-K k-mers por document frequency (DF) |
| 5 | Construcción de la matriz sparse CSR (presencia/ausencia binaria) |
| 6 | EDA de la matriz: sparsidad, distribuciones, calidad |
| 7 | Guardado de `matriz_kmers.npz`, `kmer_vocab.npy`, `ids_kmers.json` |

**Input:**
- `datos/brutos/genomas_fasta/` — archivos `.fasta` / `.fa` / `.fna` de los ensamblajes WGS
- `datos/procesados/target_metadata.csv` — genome_id y etiquetas (salida del notebook 03)

**Output:**
- `datos/procesados/matriz_kmers.npz` — matriz sparse CSR (N × top_K), valores binarios 0/1
- `datos/procesados/kmer_vocab.npy` — array de strings con los k-mers seleccionados (top_K,)
- `datos/procesados/ids_kmers.json` — lista ordenada de genome_ids (alineada con las filas)
- `datos/procesados/informe_kmers.json` — métricas reproducibles del proceso
- `figuras/04_*.png` — figuras EDA de la matriz k-mer

---
### Decisiones de diseño — por qué k=31 y por qué DF

**k=31:**  
- Odd → evita k-mers palindrómicos que se solapan con su complemento reverso.  
- 31 es el estándar *de facto* en genómica bacteriana: suficiente especificidad
  (4³¹ ≈ 4.6 × 10¹⁸ posibles k-mers >> tamaño genoma ~5 Mb) sin fragmentar
  señal biológica útil.  
- Replicamos el setup de Moradigaravand et al. (2018).

**Document Frequency (DF) en lugar de Term Frequency (TF):**  
- DF = número de genomas en los que aparece un k-mer (≥ 1 vez).  
- K-mers con DF muy baja (aparecen en 1-2 genomas) son ruido/errores de
  secuenciación, no señal biológica generalizable.  
- K-mers con DF muy alta (~100% de genomas) son conservados en todos los
  fondos genéticos → no discriminantes para AMR.  
- Seleccionar los **top-K por DF** filtrando ambos extremos maximiza la
  señal informativa para el clasificador.

**Binarización (presencia/ausencia):**  
- La frecuencia bruta de un k-mer varía con la cobertura y la longitud del
  ensamblaje, introduciendo sesgo técnico.  
- La representación binaria es más robusta y es la usada en la literatura
  AMR-ML (Moradigaravand 2018, Nguyen 2019).

> **Dependencia externa:** Este notebook requiere **jellyfish ≥ 2.x** instalado
> en el sistema. Si no está disponible, la Celda 1 mostrará las instrucciones
> de instalación y abortará con un mensaje claro.

---
## Celda 0 — Imports y configuración global

In [ ]:
# ─── Librería estándar ────────────────────────────────────────────────────────
import json
import logging
import shutil
import subprocess
import tempfile
from pathlib import Path
from datetime import datetime

# ─── Third-party ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import psutil
from scipy.sparse import csr_matrix, save_npz, load_npz
from tqdm.notebook import tqdm

# ─── Logging ──────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ─── Rutas del proyecto ───────────────────────────────────────────────────────
ROOT        = Path.cwd().parent
DIR_FASTA   = ROOT / "datos" / "brutos" / "genomas_fasta"
DIR_PROC    = ROOT / "datos" / "procesados"
DIR_FIGS    = ROOT / "figuras"
DIR_TMP     = ROOT / "datos" / "procesados" / "jellyfish_tmp"

for d in [DIR_PROC, DIR_FIGS, DIR_TMP]:
    d.mkdir(parents=True, exist_ok=True)

# ─── Parámetros k-mer ─────────────────────────────────────────────────────────
K       = 31        # longitud del k-mer
TOP_K   = 100_000   # k-mers más frecuentes a retener (por document frequency)
MIN_DF  = 2         # DF mínima: k-mer debe aparecer en al menos 2 genomas

# ─── Auto-detección de recursos ───────────────────────────────────────────────
# jellyfish usa ~3-4× su hash_size en memoria real (reader + buffers de threads).
# Con RAM ajustada, se prefiere menos paralelismo y hash más pequeño.
_ram_gb  = psutil.virtual_memory().available / 1e9
_n_cores = psutil.cpu_count(logical=False) or 1

# Conservador: reservar máx. 15% de RAM disponible por worker,
# sin superar 512M ni bajar de 100M.
N_JOBS    = 1 if _ram_gb < 2.0 else min(4, _n_cores)
_hash_mb  = max(100, min(512, int(_ram_gb * 150 / N_JOBS)))
HASH_SIZE = f"{_hash_mb}M"
THREADS   = 1   # 1 hilo por worker jellyfish (joblib gestiona el paralelismo externo)

log.info(f"RAM disponible : {_ram_gb:.1f} GB  |  Cores físicos : {_n_cores}")
log.info(f"HASH_SIZE/job  : {HASH_SIZE}  |  N_JOBS : {N_JOBS}")
if _ram_gb < 2.0:
    log.warning("RAM < 2 GB disponible — N_JOBS=1 para evitar OOM. El conteo será más lento.")

# ─── Estilo gráfico ───────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

print(f"Proyecto raíz : {ROOT}")
print(f"Timestamp     : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Parámetros    : k={K}  top_K={TOP_K:,}  min_DF={MIN_DF}  N_JOBS={N_JOBS}  HASH_SIZE={HASH_SIZE}")


---
## Celda 1 — Verificación de dependencias y datos de entrada

Comprobamos que:
1. `jellyfish` está instalado y accesible en el PATH.
2. Los archivos FASTA existen para los `genome_id` del target.
3. Identificamos genomas sin FASTA (se excluirán con aviso).

In [ ]:
# ── 1a. Verificar jellyfish ───────────────────────────────────────────────────
jf_path = shutil.which("jellyfish")

if jf_path is None:
    raise EnvironmentError(
        "jellyfish no encontrado en el PATH.\n"
        "Instalación:\n"
        "  conda install -c bioconda jellyfish   # recomendado\n"
        "  # o bien: sudo apt-get install jellyfish  (Ubuntu/WSL2)"
    )

version_out = subprocess.run(
    ["jellyfish", "--version"], capture_output=True, text=True
).stdout.strip()
log.info(f"jellyfish encontrado: {jf_path}  versión: {version_out}")

# ── 1b. Cargar target_metadata → lista de genome_ids válidos ──────────────────
df_meta = pd.read_csv(
    DIR_PROC / "target_metadata.csv",
    dtype={"genome_id": str}
)
ids_target = df_meta["genome_id"].tolist()
log.info(f"IDs en target_metadata: {len(ids_target)}")

# ── 1c. Inventario de FASTAs disponibles ──────────────────────────────────────
fasta_disponibles = {}
for ext in ["*.fasta", "*.fa", "*.fna"]:
    for p in DIR_FASTA.glob(ext):
        gid = p.stem
        fasta_disponibles[gid] = p

log.info(f"FASTAs encontrados en disco: {len(fasta_disponibles)}")

# ── 1d. Cruce: IDs con FASTA + IDs sin FASTA ──────────────────────────────────
ids_con_fasta    = [gid for gid in ids_target if gid in fasta_disponibles]
ids_sin_fasta    = [gid for gid in ids_target if gid not in fasta_disponibles]
ids_fasta_extra  = [gid for gid in fasta_disponibles if gid not in set(ids_target)]

print(f"\n{'─'*55}")
print(f"  IDs en target (notebook 03)    : {len(ids_target):>6}")
print(f"  IDs con FASTA disponible       : {len(ids_con_fasta):>6}")
print(f"  IDs sin FASTA (se excluirán)   : {len(ids_sin_fasta):>6}")
print(f"  FASTAs sin label (no se usan)  : {len(ids_fasta_extra):>6}")
print(f"{'─'*55}")

if ids_sin_fasta:
    log.warning(
        f"{len(ids_sin_fasta)} genomas sin FASTA. "
        "Verifica que la descarga del notebook 01 esté completa."
    )

ids_procesables = ids_con_fasta
df_meta_proc    = df_meta[df_meta["genome_id"].isin(set(ids_procesables))].copy()
df_meta_proc    = df_meta_proc.set_index("genome_id").loc[ids_procesables].reset_index()

print(f"\n  Genomas a procesar (k-mer)     : {len(ids_procesables):>6}")
print(f"  Distribución: S={( df_meta_proc['label']==0 ).sum()}  R={( df_meta_proc['label']==1 ).sum()}")

# ── 1e. Validar integridad de FASTAs ──────────────────────────────────────────
# Un FASTA corrupto produce una fila de ceros en la matriz sin ningún aviso.
BASES_VALIDAS = set("ACGTNRYSWKMBDHVacgtnryswkmbdhv")

def validar_fasta(path: Path) -> tuple[bool, str]:
    """Comprueba cabecera '>' y que la secuencia solo contiene bases IUPAC."""
    try:
        tiene_header = False
        tiene_seq    = False
        with open(path, "r") as fh:
            for line in fh:
                line = line.strip()
                if not line:
                    continue
                if line.startswith(">"):
                    tiene_header = True
                elif tiene_header:
                    invalidas = set(line) - BASES_VALIDAS
                    if invalidas:
                        return False, f"Caracteres inválidos: {invalidas}"
                    tiene_seq = True
        if not tiene_header:
            return False, "Sin cabecera '>'"
        if not tiene_seq:
            return False, "Sin secuencia válida"
        return True, "OK"
    except Exception as e:
        return False, str(e)

fasta_invalidos = {}
for gid in tqdm(ids_procesables, desc="Validando FASTAs", unit="archivo", leave=False):
    ok, msg = validar_fasta(fasta_disponibles[gid])
    if not ok:
        fasta_invalidos[gid] = msg

if fasta_invalidos:
    log.warning(f"{len(fasta_invalidos)} FASTAs inválidos (se excluirán):")
    for gid, msg in fasta_invalidos.items():
        log.warning(f"  {gid}: {msg}")
    ids_procesables = [gid for gid in ids_procesables if gid not in fasta_invalidos]
    df_meta_proc    = df_meta_proc[df_meta_proc["genome_id"].isin(set(ids_procesables))].copy()
    df_meta_proc    = df_meta_proc.set_index("genome_id").loc[ids_procesables].reset_index()
    print(f"  Genomas tras validación FASTA: {len(ids_procesables)}")
else:
    log.info(f"✓ Todos los FASTAs validados correctamente ({len(ids_procesables)})")


---
## Celda 2 — Función de conteo jellyfish por genoma

Definimos una función reutilizable que:
1. Ejecuta `jellyfish count` sobre un FASTA (k=31, canónico, lower-case).
2. Exporta el dump en formato texto con `jellyfish dump`.
3. Parsea el output y devuelve un `dict {kmer: count}`.

> **Flag `-C` (canónico):** jellyfish normaliza cada k-mer con su complemento reverso
> tomando el lexicográficamente menor. Así cada posición genómica produce un único
> k-mer independientemente de la hebra, lo que reduce el vocabulario ~2× y elimina
> redundancia biológica.

In [ ]:
def contar_kmers_jellyfish(
    fasta_path: Path,
    k: int = 31,
    threads: int = 1,
    hash_size: str = "1G",
    tmp_dir: Path = None,
) -> dict[str, int]:
    """
    Cuenta k-mers canónicos en un ensamblaje FASTA usando jellyfish.

    Parámetros
    ----------
    fasta_path : Path
        Ruta al archivo FASTA (puede ser multi-contig).
    k : int
        Longitud del k-mer (default=31).
    threads : int
        Hilos paralelos para jellyfish count (usar 1 cuando se paraleliza con joblib).
    hash_size : str
        Tamaño inicial de la tabla hash (e.g. '1G', '500M').
    tmp_dir : Path
        Directorio para archivos temporales. Si None usa tempfile.

    Retorna
    -------
    dict[str, int] : {kmer_string: frequency}
    """
    use_tmp = tmp_dir is not None
    tmpdir_ctx = tmp_dir if use_tmp else Path(tempfile.mkdtemp())

    genome_stem = fasta_path.stem
    jf_db   = tmpdir_ctx / f"{genome_stem}.jf"
    jf_dump = tmpdir_ctx / f"{genome_stem}.dump"

    try:
        # ── Paso 1: jellyfish count ────────────────────────────────────────────
        cmd_count = [
            "jellyfish", "count",
            "-C",
            "-m", str(k),
            "-s", hash_size,
            "-t", str(threads),
            "-o", str(jf_db),
            str(fasta_path),
        ]
        result = subprocess.run(cmd_count, capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError(
                f"jellyfish count falló en {fasta_path.name}:\n{result.stderr}"
            )

        # ── Paso 2: jellyfish dump (formato columnar: kmer count) ─────────────
        cmd_dump = [
            "jellyfish", "dump",
            "-c",
            "-o", str(jf_dump),
            str(jf_db),
        ]
        result = subprocess.run(cmd_dump, capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError(
                f"jellyfish dump falló en {fasta_path.name}:\n{result.stderr}"
            )

        # ── Paso 3: parsear el dump columnar con validación ───────────────────
        BASES_CANONICAS = set("ACGTacgt")
        kmer_counts = {}
        with open(jf_dump, "r") as fh:
            for line in fh:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                if len(parts) == 2 and parts[1].isdigit():
                    kmer, count = parts[0], int(parts[1])
                    # Solo bases canónicas (filtra k-mers con N u otros caracteres)
                    if set(kmer) <= BASES_CANONICAS:
                        kmer_counts[kmer] = count

        return kmer_counts

    finally:
        for f in [jf_db, jf_dump]:
            if f.exists():
                f.unlink()


# ── Test rápido con el primer genoma disponible ───────────────────────────────
gid_test   = ids_procesables[0]
fasta_test = fasta_disponibles[gid_test]

print(f"Test jellyfish → {fasta_test.name}")
kmers_test = contar_kmers_jellyfish(
    fasta_test, k=K, threads=THREADS, hash_size=HASH_SIZE, tmp_dir=DIR_TMP
)
print(f"  K-mers únicos (k={K}): {len(kmers_test):,}")
print(f"  Ejemplo (5 primeros): {dict(list(kmers_test.items())[:5])}")
print("✓ jellyfish funciona correctamente")


---
## Celda 3 — Conteo de k-mers para todos los genomas

Iteramos sobre los `N` genomas procesables y construimos dos estructuras:
- `all_kmer_dicts`: lista de dicts `{kmer: count}` por genoma.
- `doc_freq`: contador acumulado de cuántos genomas contienen cada k-mer.

**Nota de rendimiento:** Para 500 genomas (~5 Mb cada uno, k=31) este loop
tarda entre 10-30 minutos según el hardware. Es un proceso I/O intensivo,
no de CPU. Si se necesita acelerar: aumentar `THREADS` o usar `joblib.Parallel`
(ver comentario al final de la celda).

In [ ]:
from collections import Counter
from joblib import Parallel, delayed

# ── Función wrapper segura para paralelismo ────────────────────────────────────
# Cada worker usa su propio subdirectorio tmp para evitar colisiones de archivos.
def _contar_genoma(gid: str) -> tuple[str, dict]:
    tmp_worker = DIR_TMP / gid
    tmp_worker.mkdir(exist_ok=True)
    try:
        kd = contar_kmers_jellyfish(
            fasta_disponibles[gid],
            k=K,
            threads=THREADS,
            hash_size=HASH_SIZE,
            tmp_dir=tmp_worker,
        )
        return gid, kd
    except Exception as e:
        log.error(f"Error en {gid}: {e}")
        return gid, {}
    finally:
        shutil.rmtree(tmp_worker, ignore_errors=True)

# ── Conteo paralelo ────────────────────────────────────────────────────────────
log.info(f"Iniciando conteo k-mer para {len(ids_procesables)} genomas (N_JOBS={N_JOBS})...")
t_inicio = datetime.now()

resultados = Parallel(n_jobs=N_JOBS, prefer="processes", verbose=0)(
    delayed(_contar_genoma)(gid)
    for gid in tqdm(ids_procesables, desc="jellyfish count", unit="genoma")
)

t_fin    = datetime.now()
duracion = (t_fin - t_inicio).total_seconds()

# ── Reconstruir en el orden canónico de ids_procesables ───────────────────────
resultado_map  = {gid: kd for gid, kd in resultados}
all_kmer_dicts = [resultado_map[gid] for gid in ids_procesables]
errores        = [gid for gid in ids_procesables if not resultado_map[gid]]

# ── Validación de alineación post-loop ────────────────────────────────────────
assert len(all_kmer_dicts) == len(ids_procesables), (
    f"❌ Desalineación post-loop: {len(all_kmer_dicts)} dicts ≠ {len(ids_procesables)} IDs"
)
log.info("✓ Alineación all_kmer_dicts / ids_procesables verificada")

# ── Acumular document frequency ───────────────────────────────────────────────
doc_freq = Counter()
for kd in all_kmer_dicts:
    doc_freq.update(kd.keys())

n_ok        = len(ids_procesables) - len(errores)
vocab_bruto = len(doc_freq)

print(f"\n{'─'*55}")
print(f"  Genomas procesados OK   : {n_ok:>6}")
print(f"  Genomas con error       : {len(errores):>6}")
print(f"  Vocabulario bruto       : {vocab_bruto:>10,} k-mers únicos")
print(f"  Tiempo total            : {duracion:.1f} s  ({duracion/n_ok:.2f} s/genoma)")
print(f"{'─'*55}")

if errores:
    log.warning(f"Genomas con error: {errores}")


---
## Celda 4 — Vocabulario: Document Frequency y selección top-K

Analizamos la distribución de DF y seleccionamos los `TOP_K` k-mers más informativos.

**Criterios de filtrado:**
1. `DF < MIN_DF` → descartados (muy probablemente errores de secuenciación).
2. De los restantes, ordenamos por DF desc y tomamos los `TOP_K`.
   - No aplicamos filtro de DF máximo explícito: los k-mers ubicuos (DF~N)
     tendrán varianza casi cero y VarianceThreshold los eliminará en el notebook 05.

In [ ]:
N = len(ids_procesables)  # número efectivo de genomas

# ── 4a. Array de (kmer, df) ordenado por df descendente ───────────────────────
df_array = np.array(doc_freq.most_common(), dtype=object)
kmers_all = df_array[:, 0].astype(str)
dfs_all   = df_array[:, 1].astype(int)

# ── 4b. Filtro MIN_DF ──────────────────────────────────────────────────────────
mask_min_df     = dfs_all >= MIN_DF
kmers_filtrados = kmers_all[mask_min_df]
dfs_filtrados   = dfs_all[mask_min_df]
n_descartados_df = (~mask_min_df).sum()

# ── 4c. Selección top-K ───────────────────────────────────────────────────────
top_k_real = min(TOP_K, len(kmers_filtrados))
kmer_vocab = kmers_filtrados[:top_k_real]
kmer_dfs   = dfs_filtrados[:top_k_real]

# ── 4d. Cascada de filtros (tabla de reducción de vocabulario) ────────────────
print(f"\nCascada de reducción del vocabulario:")
print(f"  Vocabulario bruto              : {len(kmers_all):>10,}  (100.0%)")
print(f"  Tras MIN_DF ≥ {MIN_DF:<2}              : {len(kmers_filtrados):>10,}  ({100*len(kmers_filtrados)/len(kmers_all):.1f}%)  [-{n_descartados_df:,} singletons]")
print(f"  Tras top-{TOP_K//1000}k selección     : {top_k_real:>10,}  ({100*top_k_real/len(kmers_all):.1f}%)")
print(f"\nEstadísticas DF del vocabulario final:")
print(f"  DF mínima  : {kmer_dfs.min():,}")
print(f"  DF mediana : {int(np.median(kmer_dfs)):,}")
print(f"  DF máxima  : {kmer_dfs.max():,}  ({100*kmer_dfs.max()/N:.1f}% de los genomas)")

# ── 4e. Figura: distribución de Document Frequency ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.hist(dfs_all, bins=50, color="steelblue", edgecolor="white", linewidth=0.4)
ax.set_yscale("log")
ax.axvline(MIN_DF, color="tomato", linestyle="--", lw=1.5, label=f"MIN_DF={MIN_DF}")
ax.set_xlabel("Document Frequency")
ax.set_ylabel("Número de k-mers (log)")
ax.set_title(f"Distribución DF — vocabulario bruto ({vocab_bruto:,} k-mers)")
ax.legend()

ax = axes[1]
ax.hist(kmer_dfs, bins=50, color="mediumseagreen", edgecolor="white", linewidth=0.4)
ax.axvline(np.median(kmer_dfs), color="darkorange", linestyle="--",
           lw=1.5, label=f"mediana={int(np.median(kmer_dfs))}")
ax.set_xlabel("Document Frequency")
ax.set_ylabel("Número de k-mers")
ax.set_title(f"DF — top-{top_k_real:,} k-mers seleccionados")
ax.legend()

plt.tight_layout()
fig.savefig(DIR_FIGS / "04_document_frequency.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figura guardada → 04_document_frequency.png")


In [ ]:
# ── Análisis de sensibilidad: TOP_K × MIN_DF ──────────────────────────────────
# Justifica los valores elegidos mostrando cuánto vocabulario retiene cada
# combinación. Permite detectar si TOP_K=100k es conservador o demasiado agresivo.

top_k_candidatos  = [25_000, 50_000, 100_000, 200_000, 500_000]
min_df_candidatos = [2, 5, 10, 20]

print("Vocabulario retenido por combinación TOP_K × MIN_DF:")
header = f"{'':>16}" + "".join(f"  DF≥{mdf:>2}" for mdf in min_df_candidatos)
print(header)
print("─" * len(header))

for top_k_c in top_k_candidatos:
    row = f"  TOP_K={top_k_c:>7,}"
    for mdf in min_df_candidatos:
        n_pass = min(top_k_c, int((dfs_all >= mdf).sum()))
        row += f"  {n_pass:>6,}"
    print(row)

print(f"\n  ► Valor seleccionado: TOP_K={TOP_K:,}  MIN_DF={MIN_DF}")
print(f"    Retiene {100*top_k_real/vocab_bruto:.1f}% del vocabulario bruto  |  "
      f"DF mínima en vocab: {kmer_dfs.min()}")


---
## Celda 5 — Construcción de la matriz sparse CSR

Construimos `X_kmers` de dimensiones `(N × top_K)` con valores **binarios** (0/1).

**Por qué binario y no frecuencias brutas:**
- Las frecuencias de k-mers varían con la cobertura del ensamblaje (sesgo técnico).
- La binarización hace la representación independiente de la cobertura.
- Es la representación estándar en la literatura AMR-ML y da mejores resultados
  en clasificadores de árbol (RF, XGBoost) según Moradigaravand et al. (2018).

**Por qué sparse (CSR):**
- En una matriz N×100k binaria, la mayoría de entradas son 0.
  Un genoma típico de 5 Mb con k=31 produce ~160k k-mers únicos de un vocabulario
  de 100k → densidad ~1-5%. La representación sparse ahorra >95% de memoria.

In [ ]:
# ── 5a. Índice kmer→columna para lookup O(1) ──────────────────────────────────
kmer_to_col = {kmer: col for col, kmer in enumerate(kmer_vocab)}
vocab_set   = set(kmer_vocab)   # para filtrado rápido

# ── 5b. Construir sparse matrix en formato COO (lil_matrix más lento) ─────────
# Usamos listas de (fila, col) para construir CSR eficientemente
rows_idx = []
cols_idx = []

for i, kd in enumerate(tqdm(all_kmer_dicts, desc="Construyendo matriz", unit="genoma")):
    for kmer in kd:          # iteramos sólo sobre k-mers presentes en este genoma
        if kmer in kmer_to_col:
            rows_idx.append(i)
            cols_idx.append(kmer_to_col[kmer])
            # Valor = 1 (binario); no añadimos duplicados porque jellyfish ya
            # devuelve k-mers únicos por genoma

# ── 5c. Crear matriz CSR ──────────────────────────────────────────────────────
data     = np.ones(len(rows_idx), dtype=np.int8)
X_kmers  = csr_matrix(
    (data, (rows_idx, cols_idx)),
    shape=(N, top_k_real),
    dtype=np.int8
)

# ── 5d. Estadísticas de la matriz ─────────────────────────────────────────────
densidad    = X_kmers.nnz / (X_kmers.shape[0] * X_kmers.shape[1])
memoria_mb  = (X_kmers.data.nbytes + X_kmers.indices.nbytes + X_kmers.indptr.nbytes) / 1e6
memoria_densa_mb = (N * top_k_real) / 1e6   # memoria si fuera densa (float32)

print(f"\n{'─'*55}")
print(f"  Shape              : {X_kmers.shape}")
print(f"  Elementos no cero  : {X_kmers.nnz:,}")
print(f"  Densidad           : {100*densidad:.2f}%")
print(f"  Memoria sparse     : {memoria_mb:.1f} MB")
print(f"  Memoria densa (est): {memoria_densa_mb:.0f} MB  (float32)")
print(f"  Factor compresión  : {memoria_densa_mb/memoria_mb:.0f}×")
print(f"{'─'*55}")

In [ ]:
# ── Detección de genomas duplicados / clones ───────────────────────────────────
# Genomas con similitud Jaccard > 0.95 indican posibles clones que pueden sesgar
# el CV (si un clon aparece en train y test, el modelo memoriza en vez de generalizar).
#
# Cálculo eficiente: Jaccard(A,B) = |A∩B| / |A∪B|
# Con vectores binarios: intersección = producto punto, unión = |A| + |B| - intersección.
# X @ X.T opera sobre sparse → resultado (N×N) pequeño.

UMBRAL_JACCARD = 0.95

def detectar_clones(X: csr_matrix, ids: list, umbral: float = 0.95):
    interseccion = (X @ X.T).toarray().astype(float)  # (N, N)
    card = np.array(X.sum(axis=1)).flatten()
    union = card[:, None] + card[None, :] - interseccion
    with np.errstate(divide="ignore", invalid="ignore"):
        J = np.where(union > 0, interseccion / union, 0.0)
    np.fill_diagonal(J, 0.0)
    filas, cols = np.where(J > umbral)
    pares = [(int(i), int(j), float(J[i, j])) for i, j in zip(filas, cols) if i < j]
    return sorted(pares, key=lambda x: -x[2])

log.info(f"Calculando similitud Jaccard entre {N} genomas...")
pares_clones = detectar_clones(X_kmers, ids_procesables, UMBRAL_JACCARD)

if pares_clones:
    log.warning(f"{len(pares_clones)} pares con Jaccard > {UMBRAL_JACCARD}:")
    for i, j, sim in pares_clones[:10]:
        yi, yj = ("S" if y[i] == 0 else "R"), ("S" if y[j] == 0 else "R")
        print(f"  [{yi}] {ids_procesables[i]}  ↔  [{yj}] {ids_procesables[j]}  J={sim:.3f}")
    if len(pares_clones) > 10:
        print(f"  ... y {len(pares_clones)-10} pares más")
    print("\n  ⚠ Revisar estos pares antes del entrenamiento.")
    print("  En notebook 06, StratifiedKFold puede separarlos en train/test diferentes.")
else:
    log.info(f"✓ Sin clones detectados (Jaccard > {UMBRAL_JACCARD}) entre {N} genomas.")


---
## Celda 6 — EDA de la matriz k-mer

Antes de guardar, hacemos tres comprobaciones clave:
1. **Distribución de k-mers por genoma** — detecta outliers (genomas muy pobres o muy ricos).
2. **Heatmap de co-ocurrencia S vs R** — ¿existen k-mers con señal discriminante visual?
3. **Alineación con y_final** — verificación crítica de que el orden de filas es correcto.

In [ ]:
# ── 6a. Carga de y_final alineado ─────────────────────────────────────────────
y_full    = np.load(DIR_PROC / "y_final.npy")
meta_full = pd.read_csv(DIR_PROC / "target_metadata.csv", dtype={"genome_id": str})

# ALINEACIÓN: df_meta_proc está ordenado igual que all_kmer_dicts (ids_procesables)
y = df_meta_proc["label"].to_numpy(dtype=np.int8)

assert len(y) == X_kmers.shape[0], (
    f"❌ Desalineación: y tiene {len(y)} muestras, X_kmers {X_kmers.shape[0]} filas"
)
log.info(f"✓ Alineación y / X_kmers verificada ({len(y)} muestras)")

# ── 6b. K-mers por genoma (número de columnas activas por fila) ────────────────
kmers_por_genoma = np.diff(X_kmers.indptr)   # equivalente a X_kmers.getnnz(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histograma por clase
ax = axes[0]
for label, color, nombre in [(0, "steelblue", "Susceptible"), (1, "tomato", "Resistente")]:
    ax.hist(
        kmers_por_genoma[y == label],
        bins=40, alpha=0.7, label=nombre, color=color, edgecolor="white"
    )
ax.set_xlabel(f"K-mers en vocabulario (top-{top_k_real:,}) presentes por genoma")
ax.set_ylabel("Número de genomas")
ax.set_title("Distribución de k-mers activos por genoma")
ax.legend()

# Boxplot por clase
ax = axes[1]
df_box = pd.DataFrame({
    "n_kmers": kmers_por_genoma,
    "clase"  : ["Susceptible" if yi == 0 else "Resistente" for yi in y]
})
sns.boxplot(data=df_box, x="clase", y="n_kmers", palette=["steelblue", "tomato"], ax=ax)
ax.set_xlabel("")
ax.set_ylabel("K-mers activos")
ax.set_title("K-mers activos por clase (boxplot)")

plt.tight_layout()
fig.savefig(DIR_FIGS / "04_kmers_por_genoma.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figura guardada → 04_kmers_por_genoma.png")

# ── Diagnóstico: outliers de k-mers por genoma ────────────────────────────────
p1, p99 = np.percentile(kmers_por_genoma, [1, 99])
outliers = np.where((kmers_por_genoma < p1) | (kmers_por_genoma > p99))[0]
if len(outliers) > 0:
    log.warning(
        f"{len(outliers)} genomas outlier (k-mers fuera de [p1={p1:.0f}, p99={p99:.0f}]). "
        "Verificar si son ensamblajes de mala calidad que pasaron el QC."
    )
    for idx in outliers[:5]:
        print(f"  Genoma idx={idx}  id={ids_procesables[idx]}  k-mers={kmers_por_genoma[idx]}")
else:
    print("✓ Sin outliers extremos en k-mers por genoma")

In [ ]:
# ── 6c. Frecuencia media de cada k-mer en S vs R ──────────────────────────────
# Para los top-500 k-mers (más DF): ¿hay señal discriminante?

TOP_PLOT = 500   # sólo visualizamos los 500 más frecuentes

X_top500 = X_kmers[:, :TOP_PLOT].toarray().astype(float)

mean_S   = X_top500[y == 0].mean(axis=0)   # frecuencia media en Susceptibles
mean_R   = X_top500[y == 1].mean(axis=0)   # frecuencia media en Resistentes
diff_SR  = mean_R - mean_S                 # diferencia de medias (R − S)

fig, ax = plt.subplots(figsize=(13, 4))
colors = ["tomato" if d > 0 else "steelblue" for d in diff_SR]
ax.bar(range(TOP_PLOT), diff_SR, color=colors, width=1.0, linewidth=0)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel(f"K-mer (top-{TOP_PLOT} por DF, ordenados por DF desc)")
ax.set_ylabel("Diferencia de frecuencia media (R − S)")
ax.set_title(
    f"Señal discriminante k-mer: rojo = más frecuente en R, azul = más frecuente en S"
)
plt.tight_layout()
fig.savefig(DIR_FIGS / "04_discriminancia_SR.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figura guardada → 04_discriminancia_SR.png")

# Top-10 k-mers más diferenciales
top10_idx = np.argsort(np.abs(diff_SR))[::-1][:10]
print("\nTop-10 k-mers más discriminantes (por |diferencia R−S|):")
for rank, idx in enumerate(top10_idx, 1):
    print(f"  {rank:2}. {kmer_vocab[idx]}  diff={diff_SR[idx]:+.3f}  DF={kmer_dfs[idx]:,}")

---
## Celda 7 — Guardado de artefactos y verificación

Guardamos:
- `matriz_kmers.npz` — matriz sparse CSR lista para el notebook 05.
- `kmer_vocab.npy` — vocabulario (los k-mers como strings).
- `ids_kmers.json` — lista de genome_ids en el mismo orden que las filas.
- `informe_kmers.json` — métricas reproducibles.

> **IMPORTANTE — alineación garantizada:** El orden de `ids_kmers.json` corresponde
> exactamente al orden de filas de `matriz_kmers.npz` y al de `y` generado en esta
> sesión. El notebook 05 debe cargar ambos juntos y verificar esta alineación.

In [ ]:
# ── 7a. Guardar matriz sparse ─────────────────────────────────────────────────
ruta_matriz = DIR_PROC / "matriz_kmers.npz"
save_npz(ruta_matriz, X_kmers)
log.info(f"matriz_kmers.npz guardado → {ruta_matriz}  shape={X_kmers.shape}  nnz={X_kmers.nnz:,}")

# ── 7b. Guardar vocabulario k-mer ─────────────────────────────────────────────
ruta_vocab = DIR_PROC / "kmer_vocab.npy"
np.save(ruta_vocab, kmer_vocab)
log.info(f"kmer_vocab.npy guardado → {ruta_vocab}  shape={kmer_vocab.shape}")

# ── 7c. Guardar lista de IDs (orden canónico de filas) ────────────────────────
ruta_ids = DIR_PROC / "ids_kmers.json"
with open(ruta_ids, "w") as f:
    json.dump(ids_procesables, f, indent=2)
log.info(f"ids_kmers.json guardado → {ruta_ids}")

# ── 7d. Guardar y_kmers.npy ───────────────────────────────────────────────────
ruta_y_kmers = DIR_PROC / "y_kmers.npy"
np.save(ruta_y_kmers, y)
log.info(f"y_kmers.npy guardado → {ruta_y_kmers}  shape={y.shape}")

# ── 7e. Informe JSON ──────────────────────────────────────────────────────────
informe = {
    "timestamp"             : datetime.now().isoformat(),
    "parametros": {
        "k"                 : K,
        "top_K"             : top_k_real,
        "min_df"            : MIN_DF,
        "threads_jellyfish" : THREADS,
        "n_jobs"            : N_JOBS,
        "hash_size_por_job" : HASH_SIZE,
        "binarizacion"      : True,
    },
    "genomas": {
        "n_input"           : len(ids_target),
        "n_sin_fasta"       : len(ids_sin_fasta),
        "n_fasta_invalidos" : len(fasta_invalidos),
        "n_error_jellyfish" : len(errores),
        "n_final"           : N,
        "n_susceptible"     : int((y == 0).sum()),
        "n_resistente"      : int((y == 1).sum()),
    },
    "vocabulario": {
        "n_kmers_bruto"     : vocab_bruto,
        "n_kmers_mindf"     : int(mask_min_df.sum()),
        "n_kmers_final"     : top_k_real,
        "pct_retenido"      : round(100 * top_k_real / vocab_bruto, 2),
    },
    "matriz": {
        "shape"             : list(X_kmers.shape),
        "nnz"               : int(X_kmers.nnz),
        "densidad_pct"      : round(100 * densidad, 3),
        "memoria_sparse_mb" : round(memoria_mb, 1),
    },
    "clones": {
        "umbral_jaccard"    : UMBRAL_JACCARD,
        "n_pares_detectados": len(pares_clones),
    },
    "archivos": {
        "matriz_kmers"      : str(ruta_matriz),
        "kmer_vocab"        : str(ruta_vocab),
        "ids_kmers"         : str(ruta_ids),
        "y_kmers"           : str(ruta_y_kmers),
    }
}

ruta_inf = DIR_PROC / "informe_kmers.json"
with open(ruta_inf, "w") as f:
    json.dump(informe, f, indent=2, ensure_ascii=False)

print(json.dumps(informe, indent=2, ensure_ascii=False))
log.info(f"Informe guardado → {ruta_inf}")


In [ ]:
# ── 7f. Verificación de integridad post-guardado ──────────────────────────────
X_check     = load_npz(DIR_PROC / "matriz_kmers.npz")
vocab_check = np.load(DIR_PROC / "kmer_vocab.npy", allow_pickle=True)
y_check     = np.load(DIR_PROC / "y_kmers.npy")

with open(DIR_PROC / "ids_kmers.json") as f:
    ids_check = json.load(f)

assert X_check.shape[0] == len(ids_check),   "❌ Filas X ≠ longitud ids"
assert X_check.shape[1] == len(vocab_check), "❌ Columnas X ≠ longitud vocab"
assert X_check.shape[0] == len(y_check),     "❌ Filas X ≠ longitud y"
assert X_check.dtype     == np.int8,         "❌ dtype inesperado"

print("── Verificación post-guardado ───────────────────────")
print(f"  matriz_kmers.npz  : shape={X_check.shape}  dtype={X_check.dtype}  nnz={X_check.nnz:,}")
print(f"  kmer_vocab.npy    : {len(vocab_check):,} k-mers")
print(f"  ids_kmers.json    : {len(ids_check)} genome_ids")
print(f"  y_kmers.npy       : shape={y_check.shape}  S={( y_check==0 ).sum()}  R={( y_check==1 ).sum()}")
print("")
print("✓ Todos los archivos correctamente alineados")
print(f"\n{'='*50}")
print("RESUMEN DEL NOTEBOOK 04")
print(f"{'='*50}")
print(f"  Genomas procesados       : {N}")
print(f"  Vocabulario k-mer (k={K}) : {top_k_real:,} k-mers")
print(f"  Densidad de la matriz    : {100*densidad:.2f}%")
print(f"  Memoria sparse           : {memoria_mb:.1f} MB")
print(f"  Distribución: S={int((y_check==0).sum())}  R={int((y_check==1).sum())}")
print(f"\n  Siguiente paso → notebook 05: integración genes AMR + selección de features")